In [7]:
pip install flask lightgbm pandas scikit-learn matplotlib seaborn joblib


   -------------------------- ------------- 2/3 [flask]
   ---------------------------------------- 3/3 [flask]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# ==========================
# SIMPLIFIED LIGHTGBM AutoML Flask App
# ==========================

import subprocess
import sys

def install_package(package):
    """Install a package if not already installed"""
    try:
        __import__(package)
        print(f"✅ {package} is already installed")
    except ImportError:
        print(f"⚠️  Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

# Install required packages
required_packages = [
    'flask',
    'lightgbm',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'joblib'
]

print("="*60)
print("CHECKING AND INSTALLING REQUIRED PACKAGES")
print("="*60)

for package in required_packages:
    install_package(package)

print("\n" + "="*60)
print("ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("="*60 + "\n")

# Now import the packages
from flask import Flask, render_template, request, jsonify, send_from_directory
import pandas as pd
import joblib
import os
import webbrowser
from threading import Timer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import numpy as np
import logging
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, accuracy_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Initialize Flask app
app = Flask(__name__)

# Configuration
DATA_PATH = r'D:\2026 Work\Dr Pavitar Singh\Paper 2\Data 2\Data 2.csv'
MODEL_DIR = 'saved_models'
PLOTS_DIR = 'static/plots'
TEMPLATE_DIR = 'templates'
LOG_FILE = 'app.log'

# LightGBM Hyperparameters
LIGHTGBM_PARAMS = {
    'learning_rate': 0.05,
    'max_depth': 3,
    'min_data_in_leaf': 10,
    'n_estimators': 100,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'reg_alpha': 0.0,
    'reg_lambda': 0.0,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'min_child_weight': 0.001,
    'min_split_gain': 0.0,
    'importance_type': 'gain'
}

# Setup directories
def setup_directories():
    """Create necessary directories"""
    for directory in [MODEL_DIR, PLOTS_DIR, TEMPLATE_DIR]:
        os.makedirs(directory, exist_ok=True)
        print(f"✅ Created directory: {directory}")
    
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler()]
    )

# Global variables
model = None
feature_names = []
feature_info = {}
target_column = ""
problem_type = "regression"
model_date = "Not trained"
dataset_name = ""
scaler = None
label_encoders = {}
model_performance = {}

def load_and_prepare_data():
    """Load and prepare the dataset"""
    global feature_names, feature_info, target_column, problem_type, dataset_name
    
    print(f"\n📁 Loading dataset: {DATA_PATH}")
    
    if not os.path.exists(DATA_PATH):
        print(f"❌ ERROR: File not found: {DATA_PATH}")
        print(f"   Please check the path and try again")
        return False
    
    try:
        # Load data with different encodings
        encodings = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
        df = None
        
        for encoding in encodings:
            try:
                df = pd.read_csv(DATA_PATH, encoding=encoding)
                print(f"   ✅ Loaded with {encoding} encoding")
                break
            except:
                continue
        
        if df is None:
            print("❌ ERROR: Could not load file with any encoding")
            return False
        
        # Clean column names
        df.columns = [str(col).strip() for col in df.columns]
        dataset_name = os.path.basename(DATA_PATH)
        
        print(f"   📊 Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"   📋 Columns: {', '.join(df.columns.tolist())}")
        
        # Find target column
        target_candidates = []
        for col in df.columns:
            col_lower = col.lower()
            if any(target in col_lower for target in ['cs', 'compressive', 'strength', 'fc']):
                target_candidates.append(col)
        
        if target_candidates:
            target_column = target_candidates[0]
            print(f"   🎯 Target column found: '{target_column}'")
        else:
            target_column = df.columns[-1]
            print(f"   ⚠️  Target column not found, using: '{target_column}'")
        
        # Prepare features
        feature_names = [col for col in df.columns if col != target_column]
        
        # Detect problem type
        unique_targets = df[target_column].nunique()
        if unique_targets <= 10:
            problem_type = "classification"
            print(f"   🤖 Problem type: Classification ({unique_targets} classes)")
        else:
            problem_type = "regression"
            print(f"   🤖 Problem type: Regression ({unique_targets} unique values)")
        
        # Generate feature information
        feature_info = {}
        for feature in feature_names:
            col_data = df[feature].dropna()
            
            if len(col_data) == 0:
                feature_info[feature] = {'type': 'unknown', 'values': [], 'range': 'N/A'}
                continue
            
            # Detect feature type
            unique_vals = col_data.nunique()
            if unique_vals <= 15 or df[feature].dtype == 'object':
                feature_type = 'categorical'
                sample_values = col_data.astype(str).unique().tolist()[:10]
                range_info = f"{len(sample_values)} categories"
                min_val = None
                max_val = None
                mean_val = None
            else:
                feature_type = 'numerical'
                sample_values = [float(x) for x in col_data.head(5)] if col_data.dtype in [np.int64, np.float64] else [str(x) for x in col_data.head(5)]
                if col_data.dtype in [np.int64, np.float64]:
                    range_info = f"{col_data.min():.2f} to {col_data.max():.2f}"
                    min_val = float(col_data.min())
                    max_val = float(col_data.max())
                    mean_val = float(col_data.mean())
                else:
                    range_info = "text data"
                    min_val = None
                    max_val = None
                    mean_val = None
            
            feature_info[feature] = {
                'type': feature_type,
                'values': sample_values,
                'range': range_info,
                'dtype': str(df[feature].dtype),
                'min': min_val,
                'max': max_val,
                'mean': mean_val
            }
        
        print(f"   🔍 Detected {len(feature_names)} features")
        return True
        
    except Exception as e:
        print(f"❌ ERROR loading data: {str(e)}")
        return False

def train_model():
    """Train the LightGBM model"""
    global model, model_date, scaler, label_encoders, model_performance
    
    print("\n🤖 Training LightGBM model...")
    
    try:
        # Load data
        df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')
        df.columns = [str(col).strip() for col in df.columns]
        
        X = df[feature_names]
        y = df[target_column]
        
        # Preprocess data
        X_processed = X.copy()
        label_encoders = {}
        
        for feature in feature_names:
            if feature_info[feature]['type'] == 'categorical':
                le = LabelEncoder()
                X_processed[feature] = le.fit_transform(X_processed[feature].astype(str))
                label_encoders[feature] = le
            else:
                X_processed[feature] = pd.to_numeric(X_processed[feature], errors='coerce')
        
        # Handle missing values
        X_processed = X_processed.fillna(X_processed.mean(numeric_only=True))
        
        # Scale features
        scaler = StandardScaler()
        X_processed = pd.DataFrame(scaler.fit_transform(X_processed), columns=feature_names)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
        
        print(f"   📊 Train shape: {X_train.shape}, Test shape: {X_test.shape}")
        
        # Train model
        if problem_type == "classification":
            model = lgb.LGBMClassifier(**LIGHTGBM_PARAMS)
        else:
            model = lgb.LGBMRegressor(**LIGHTGBM_PARAMS)
        
        model.fit(X_train, y_train)
        
        # Evaluate
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        if problem_type == "classification":
            train_acc = accuracy_score(y_train, y_train_pred)
            test_acc = accuracy_score(y_test, y_test_pred)
            model_performance = {
                'train_accuracy': train_acc,
                'test_accuracy': test_acc
            }
            print(f"   ✅ Model trained - Accuracy: {test_acc:.4f}")
        else:
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            
            model_performance = {
                'train_r2': train_r2,
                'test_r2': test_r2,
                'train_rmse': train_rmse,
                'test_rmse': test_rmse
            }
            print(f"   ✅ Model trained - R² Score: {test_r2:.4f}")
        
        # Save model
        model_data = {
            'model': model,
            'scaler': scaler,
            'label_encoders': label_encoders,
            'feature_info': feature_info,
            'performance': model_performance
        }
        
        model_path = os.path.join(MODEL_DIR, f"lightgbm_model_{dataset_name.split('.')[0]}.pkl")
        joblib.dump(model_data, model_path)
        
        model_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"   💾 Model saved: {model_path}")
        return True
        
    except Exception as e:
        print(f"❌ ERROR training model: {str(e)}")
        return False

def load_existing_model():
    """Load existing model if available"""
    global model, scaler, label_encoders, model_performance
    
    model_path = os.path.join(MODEL_DIR, f"lightgbm_model_{dataset_name.split('.')[0]}.pkl")
    
    if os.path.exists(model_path):
        try:
            print(f"\n🔍 Looking for existing model...")
            model_data = joblib.load(model_path)
            model = model_data['model']
            scaler = model_data.get('scaler')
            label_encoders = model_data.get('label_encoders', {})
            model_performance = model_data.get('performance', {})
            print(f"   ✅ Loaded existing model from {model_path}")
            return True
        except Exception as e:
            print(f"   ⚠️  Could not load existing model: {str(e)}")
            return False
    else:
        print(f"   ℹ️  No existing model found at {model_path}")
        return False

def create_template():
    """Create HTML template"""
    print("\n📝 Creating web interface...")
    
    template_content = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>LightGBM AutoML - Compressive Strength Prediction</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        body { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); min-height: 100vh; padding: 20px; }
        .card { border-radius: 15px; box-shadow: 0 10px 30px rgba(0,0,0,0.1); border: none; margin-bottom: 20px; }
        .header { background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); color: white; padding: 20px; border-radius: 15px 15px 0 0; }
        .result { background: linear-gradient(135deg, #00b09b 0%, #96c93d 100%); color: white; padding: 20px; border-radius: 10px; }
        .form-control { border: 2px solid #e9ecef; border-radius: 8px; padding: 10px; transition: all 0.3s; }
        .form-control:focus { border-color: #3498db; box-shadow: 0 0 0 0.2rem rgba(52, 152, 219, 0.25); }
        .btn-primary { background: linear-gradient(135deg, #3498db 0%, #2c3e50 100%); border: none; padding: 12px 30px; border-radius: 8px; font-weight: bold; }
        .btn-primary:hover { transform: translateY(-2px); box-shadow: 0 5px 15px rgba(0,0,0,0.2); }
        .info-card { background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%); }
        .alert-error { background: linear-gradient(135deg, #e74c3c 0%, #c0392b 100%); color: white; }
        .badge { font-size: 0.9em; padding: 5px 10px; }
    </style>
</head>
<body>
    <div class="container">
        <div class="card">
            <div class="header">
                <h1 class="display-5"><i class="fas fa-robot"></i> LightGBM AutoML</h1>
                <p class="lead">Compressive Strength Prediction System</p>
                <div class="d-flex justify-content-between align-items-center mt-3">
                    <div>
                        <span class="badge bg-info"><i class="fas fa-database"></i> {{ dataset_name }}</span>
                        <span class="badge bg-success ms-2"><i class="fas fa-bullseye"></i> {{ target_column }}</span>
                    </div>
                    <a href="/retrain" class="btn btn-light btn-sm"><i class="fas fa-sync-alt"></i> Retrain Model</a>
                </div>
            </div>
            
            <div class="card-body">
                {% if error %}
                <div class="alert alert-error mb-4">
                    <i class="fas fa-exclamation-triangle"></i> {{ error }}
                </div>
                {% endif %}
                
                <div class="row">
                    <!-- Input Form -->
                    <div class="col-lg-8">
                        <div class="card mb-4">
                            <div class="card-body">
                                <h4 class="mb-4"><i class="fas fa-sliders-h"></i> Input Parameters</h4>
                                <form method="POST" action="/predict" id="predictionForm">
                                    <div class="row">
                                        {% for feature in feature_names %}
                                        {% set info = feature_info[feature] %}
                                        <div class="col-md-6 mb-3">
                                            <label class="form-label fw-bold">{{ feature }}</label>
                                            {% if info['type'] == 'categorical' %}
                                            <select class="form-control" name="{{ feature }}" required>
                                                <option value="">Select...</option>
                                                {% for value in info['values'] %}
                                                <option value="{{ value }}" {% if values.get(feature) == value %}selected{% endif %}>{{ value }}</option>
                                                {% endfor %}
                                            </select>
                                            {% else %}
                                            <input type="number" step="0.01" class="form-control" 
                                                   name="{{ feature }}" 
                                                   value="{{ values.get(feature, '') }}"
                                                   placeholder="{{ info['range'] }}" required>
                                            {% endif %}
                                            <small class="text-muted">
                                                <i class="fas fa-info-circle"></i> {{ info['range'] }}
                                                {% if info['type'] == 'numerical' and info['mean'] %}
                                                | Mean: {{ "%.2f"|format(info['mean']) }}
                                                {% endif %}
                                            </small>
                                        </div>
                                        {% endfor %}
                                    </div>
                                    <div class="d-flex justify-content-between mt-4">
                                        <button type="submit" class="btn btn-primary btn-lg">
                                            <i class="fas fa-bolt"></i> Predict Strength
                                        </button>
                                        <button type="reset" class="btn btn-outline-secondary">
                                            <i class="fas fa-redo"></i> Clear Form
                                        </button>
                                    </div>
                                </form>
                            </div>
                        </div>
                    </div>
                    
                    <!-- Results Panel -->
                    <div class="col-lg-4">
                        {% if result %}
                        <div class="result mb-4">
                            <h3 class="mb-3"><i class="fas fa-chart-line"></i> Prediction Result</h3>
                            <div class="display-4 fw-bold my-4">{{ result }}</div>
                            <div class="d-flex justify-content-between align-items-center">
                                <div>
                                    <i class="fas fa-clock"></i> {{ prediction_time }}
                                </div>
                                <div>
                                    <span class="badge bg-light text-dark">{{ problem_type|title }}</span>
                                </div>
                            </div>
                        </div>
                        {% endif %}
                        
                        <div class="card info-card">
                            <div class="card-body">
                                <h5 class="mb-3"><i class="fas fa-info-circle"></i> System Information</h5>
                                
                                <div class="mb-3">
                                    <p class="mb-1"><strong><i class="fas fa-database"></i> Dataset:</strong></p>
                                    <p class="text-muted">{{ dataset_name }}</p>
                                </div>
                                
                                <div class="mb-3">
                                    <p class="mb-1"><strong><i class="fas fa-bullseye"></i> Target:</strong></p>
                                    <p class="text-muted">{{ target_column }}</p>
                                </div>
                                
                                <div class="mb-3">
                                    <p class="mb-1"><strong><i class="fas fa-layer-group"></i> Features:</strong></p>
                                    <p class="text-muted">{{ feature_names|length }} variables</p>
                                </div>
                                
                                <div class="mb-3">
                                    <p class="mb-1"><strong><i class="fas fa-cogs"></i> Model:</strong></p>
                                    <p class="text-muted">LightGBM {{ problem_type|title }}</p>
                                </div>
                                
                                <div class="mb-3">
                                    <p class="mb-1"><strong><i class="fas fa-signal"></i> Status:</strong></p>
                                    {% if model %}
                                    <span class="badge bg-success">Trained & Ready</span>
                                    {% else %}
                                    <span class="badge bg-warning">Demo Mode</span>
                                    {% endif %}
                                </div>
                                
                                {% if model_performance %}
                                <hr>
                                <h6 class="mb-3"><i class="fas fa-chart-bar"></i> Performance Metrics</h6>
                                {% if problem_type == 'regression' %}
                                <div class="row">
                                    <div class="col-6 mb-2">
                                        <small>R² Score</small>
                                        <div class="fw-bold text-success">{{ "%.4f"|format(model_performance.test_r2) }}</div>
                                    </div>
                                    <div class="col-6 mb-2">
                                        <small>RMSE</small>
                                        <div class="fw-bold text-info">{{ "%.4f"|format(model_performance.test_rmse) }}</div>
                                    </div>
                                    <div class="col-6 mb-2">
                                        <small>Train R²</small>
                                        <div class="fw-bold text-secondary">{{ "%.4f"|format(model_performance.train_r2) }}</div>
                                    </div>
                                    <div class="col-6 mb-2">
                                        <small>Train RMSE</small>
                                        <div class="fw-bold text-secondary">{{ "%.4f"|format(model_performance.train_rmse) }}</div>
                                    </div>
                                </div>
                                {% else %}
                                <div class="row">
                                    <div class="col-6">
                                        <small>Test Accuracy</small>
                                        <div class="fw-bold text-success">{{ "%.2f"|format(model_performance.test_accuracy * 100) }}%</div>
                                    </div>
                                    <div class="col-6">
                                        <small>Train Accuracy</small>
                                        <div class="fw-bold text-secondary">{{ "%.2f"|format(model_performance.train_accuracy * 100) }}%</div>
                                    </div>
                                </div>
                                {% endif %}
                                {% endif %}
                                
                                <hr>
                                <div class="text-center">
                                    <small class="text-muted">
                                        <i class="fas fa-history"></i> Last trained: {{ model_date if model_date else 'Not trained' }}
                                    </small>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
                
                <!-- Feature Summary -->
                <div class="card mt-4">
                    <div class="card-body">
                        <h5><i class="fas fa-list-alt"></i> Feature Summary</h5>
                        <div class="table-responsive">
                            <table class="table table-hover">
                                <thead>
                                    <tr>
                                        <th>Feature</th>
                                        <th>Type</th>
                                        <th>Range / Values</th>
                                        <th>Data Type</th>
                                    </tr>
                                </thead>
                                <tbody>
                                    {% for feature in feature_names %}
                                    {% set info = feature_info[feature] %}
                                    <tr>
                                        <td><strong>{{ feature }}</strong></td>
                                        <td>
                                            {% if info['type'] == 'categorical' %}
                                            <span class="badge bg-info">Categorical</span>
                                            {% else %}
                                            <span class="badge bg-primary">Numerical</span>
                                            {% endif %}
                                        </td>
                                        <td>{{ info['range'] }}</td>
                                        <td><code>{{ info['dtype'] }}</code></td>
                                    </tr>
                                    {% endfor %}
                                </tbody>
                            </table>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>
    
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
    <script>
        // Form validation and styling
        document.addEventListener('DOMContentLoaded', function() {
            const form = document.getElementById('predictionForm');
            const inputs = form.querySelectorAll('input[required], select[required]');
            
            // Add validation styling
            inputs.forEach(input => {
                input.addEventListener('input', function() {
                    if (this.value.trim()) {
                        this.style.borderColor = '#28a745';
                        this.style.boxShadow = '0 0 0 0.2rem rgba(40, 167, 69, 0.25)';
                    } else {
                        this.style.borderColor = '#dc3545';
                        this.style.boxShadow = '0 0 0 0.2rem rgba(220, 53, 69, 0.25)';
                    }
                });
            });
            
            // Form submission
            form.addEventListener('submit', function(e) {
                let valid = true;
                const firstInvalid = [];
                
                inputs.forEach(input => {
                    if (!input.value.trim()) {
                        input.style.borderColor = '#dc3545';
                        input.style.boxShadow = '0 0 0 0.2rem rgba(220, 53, 69, 0.25)';
                        valid = false;
                        if (firstInvalid.length === 0) {
                            firstInvalid.push(input);
                        }
                    }
                });
                
                if (!valid) {
                    e.preventDefault();
                    alert('Please fill in all required fields!');
                    if (firstInvalid.length > 0) {
                        firstInvalid[0].focus();
                    }
                }
            });
            
            // Auto-fill sample values for demo
            const demoBtn = document.querySelector('button[type="reset"]');
            if (demoBtn) {
                demoBtn.addEventListener('click', function() {
                    setTimeout(() => {
                        inputs.forEach(input => {
                            if (input.type === 'number') {
                                const placeholder = input.placeholder || '';
                                const match = placeholder.match(/(\d+\.?\d*)/g);
                                if (match && match.length >= 2) {
                                    const min = parseFloat(match[0]);
                                    const max = parseFloat(match[match.length-1]);
                                    const avg = (min + max) / 2;
                                    input.value = avg.toFixed(2);
                                }
                            }
                        });
                    }, 100);
                });
            }
        });
        
        // Auto-focus first input
        window.onload = function() {
            const firstInput = document.querySelector('input[type="number"], select');
            if (firstInput) {
                firstInput.focus();
            }
        };
    </script>
</body>
</html>'''
    
    template_path = os.path.join(TEMPLATE_DIR, 'index.html')
    with open(template_path, 'w', encoding='utf-8') as f:
        f.write(template_content)
    
    print("   ✅ Web interface created successfully!")
    return True

# Initialize application
def initialize_app():
    """Initialize the complete application"""
    print("\n" + "="*60)
    print("🚀 INITIALIZING LIGHTGBM AUTOML SYSTEM")
    print("="*60)
    
    # Setup directories
    setup_directories()
    
    # Load data
    if not load_and_prepare_data():
        print("❌ Failed to load data. Exiting...")
        return False
    
    # Try to load existing model
    if not load_existing_model():
        # Train new model
        if not train_model():
            print("⚠️  Running in demo mode (no trained model)")
    
    # Create template
    create_template()
    
    print("\n" + "="*60)
    print("✅ INITIALIZATION COMPLETE!")
    print("="*60)
    return True

# Initialize
initialize_app()

# ======================
# FLASK ROUTES
# ======================

@app.route('/')
def index():
    """Main page"""
    template_vars = {
        'feature_names': feature_names,
        'feature_info': feature_info,
        'target_column': target_column,
        'problem_type': problem_type,
        'dataset_name': dataset_name,
        'model_performance': model_performance,
        'values': {},
        'result': None,
        'prediction_time': None,
        'error': None,
        'model_date': model_date
    }
    return render_template('index.html', **template_vars)

@app.route('/predict', methods=['POST'])
def predict():
    """Handle predictions"""
    try:
        input_values = {}
        form_values = request.form.to_dict()
        
        # Validate inputs
        for feature in feature_names:
            value = request.form.get(feature, '').strip()
            if not value:
                template_vars = {
                    'feature_names': feature_names,
                    'feature_info': feature_info,
                    'target_column': target_column,
                    'problem_type': problem_type,
                    'dataset_name': dataset_name,
                    'model_performance': model_performance,
                    'values': form_values,
                    'result': None,
                    'prediction_time': None,
                    'error': f"Missing field: {feature}",
                    'model_date': model_date
                }
                return render_template('index.html', **template_vars)
            
            if feature_info[feature]['type'] == 'numerical':
                try:
                    input_values[feature] = float(value)
                except:
                    template_vars = {
                        'feature_names': feature_names,
                        'feature_info': feature_info,
                        'target_column': target_column,
                        'problem_type': problem_type,
                        'dataset_name': dataset_name,
                        'model_performance': model_performance,
                        'values': form_values,
                        'result': None,
                        'prediction_time': None,
                        'error': f"Invalid number for {feature}: {value}",
                        'model_date': model_date
                    }
                    return render_template('index.html', **template_vars)
            else:
                input_values[feature] = value
        
        # Make prediction
        if model and scaler:
            # Prepare input
            X_input = pd.DataFrame([input_values])
            
            # Preprocess
            for feature in feature_names:
                if feature in label_encoders:
                    try:
                        X_input[feature] = label_encoders[feature].transform([input_values[feature]])[0]
                    except:
                        # Handle unseen categories
                        X_input[feature] = 0
                else:
                    X_input[feature] = float(input_values[feature])
            
            # Scale and predict
            X_input_scaled = scaler.transform(X_input)
            prediction = model.predict(X_input_scaled)[0]
            
            if problem_type == "classification":
                result = f"Class: {int(prediction)}"
            else:
                result = f"{float(prediction):.2f} MPa"
        else:
            # Demo mode
            numeric_values = [v for v in input_values.values() if isinstance(v, (int, float))]
            demo_val = sum(numeric_values) / len(numeric_values) if numeric_values else 0
            if problem_type == "classification":
                result = f"Class: {int(round(demo_val))} (Demo)"
            else:
                result = f"{demo_val:.2f} MPa (Demo)"
        
        prediction_time = datetime.now().strftime('%H:%M:%S')
        
        template_vars = {
            'feature_names': feature_names,
            'feature_info': feature_info,
            'target_column': target_column,
            'problem_type': problem_type,
            'dataset_name': dataset_name,
            'model_performance': model_performance,
            'values': form_values,
            'result': result,
            'prediction_time': prediction_time,
            'error': None,
            'model_date': model_date
        }
        return render_template('index.html', **template_vars)
    
    except Exception as e:
        print(f"Prediction error: {str(e)}")
        template_vars = {
            'feature_names': feature_names,
            'feature_info': feature_info,
            'target_column': target_column,
            'problem_type': problem_type,
            'dataset_name': dataset_name,
            'model_performance': model_performance,
            'values': request.form.to_dict(),
            'result': None,
            'prediction_time': None,
            'error': f"Error: {str(e)[:100]}",
            'model_date': model_date
        }
        return render_template('index.html', **template_vars)

@app.route('/retrain', methods=['GET'])
def retrain():
    """Retrain model"""
    global model_date
    if train_model():
        return jsonify({
            "status": "success", 
            "message": "Model retrained successfully",
            "date": model_date
        })
    else:
        return jsonify({
            "status": "error", 
            "message": "Failed to retrain model"
        }), 500

def open_browser():
    """Open browser automatically"""
    webbrowser.open_new("http://127.0.0.1:5000/")

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🌐 STARTING FLASK APPLICATION")
    print("="*60)
    print(f"\n📊 Dataset: {dataset_name}")
    print(f"🎯 Target: {target_column}")
    print(f"🔢 Features: {len(feature_names)}")
    print(f"🤖 Model: {'Trained' if model else 'Demo Mode'}")
    print(f"\n👉 Opening browser automatically...")
    print("👉 Manual access: http://127.0.0.1:5000/")
    print("="*60 + "\n")
    
    Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

CHECKING AND INSTALLING REQUIRED PACKAGES
✅ flask is already installed
✅ lightgbm is already installed
✅ pandas is already installed
⚠️  Installing scikit-learn...
✅ scikit-learn installed successfully
✅ matplotlib is already installed
✅ seaborn is already installed
✅ joblib is already installed

ALL PACKAGES INSTALLED SUCCESSFULLY!


🚀 INITIALIZING LIGHTGBM AUTOML SYSTEM
✅ Created directory: saved_models
✅ Created directory: static/plots
✅ Created directory: templates

📁 Loading dataset: D:\2026 Work\Dr Pavitar Singh\Paper 2\Data 2\Data 2.csv
   ✅ Loaded with utf-8 encoding
   📊 Dataset shape: 180 rows, 9 columns
   📋 Columns: C (Cement), MK (Metakaolin), FA (Fine Aggregate), CA (Coarse Aggregate), LD (LD Slag), SP (Superplasticizer), W (Water), DT (Days of Testing), CS (Compressive Strength)
   🎯 Target column found: 'CS (Compressive Strength)'
   🤖 Problem type: Regression (170 unique values)
   🔍 Detected 8 features

🔍 Looking for existing model...
   ✅ Loaded existing model from s

2026-01-29 19:15:55,964 - INFO - WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.101:5000
2026-01-29 19:15:55,966 - INFO - Press CTRL+C to quit
2026-01-29 19:15:57,764 - INFO - 127.0.0.1 - - [29/Jan/2026 19:15:57] "GET / HTTP/1.1" 200 -
2026-01-29 19:15:58,181 - INFO - 127.0.0.1 - - [29/Jan/2026 19:15:58] "GET /favicon.ico HTTP/1.1" 404 -
2026-01-29 19:16:20,513 - INFO - 127.0.0.1 - - [29/Jan/2026 19:16:20] "POST /predict HTTP/1.1" 200 -
